<a href="https://colab.research.google.com/github/niharnandala/ntsb-safety-review-agent/blob/main/notebooks/08_launch_jobs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q -U huggingface_hub

from google.colab import drive, userdata
from huggingface_hub import HfApi
drive.mount("/content/drive")
token = userdata.get("HF_TOKEN")
api = HfApi(token=token)
api.create_repo("niharnandala/ntsb-episodes", repo_type="dataset", private=True, exist_ok=True)
for name in ["episodes_train.jsonl", "episodes_val.jsonl", "episodes_test.jsonl"]:
    api.upload_file(path_or_fileobj=f"/content/drive/MyDrive/ntsb_agent/data/{name}",
                    path_in_repo=name, repo_id="niharnandala/ntsb-episodes", repo_type="dataset")
    print("uploaded", name)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.4.0 which is incompatible.
Mounted at /content/drive


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...data/episodes_train.jsonl:   1%|1         | 1.90MB /  169MB            

Validating                    :           |   0%            

uploaded episodes_train.jsonl
uploaded episodes_val.jsonl
uploaded episodes_test.jsonl


In [2]:
from huggingface_hub import run_uv_job
url = "https://raw.githubusercontent.com/niharnandala/ntsb-safety-review-agent/main/scripts/08_eval_job.py"
V1 = "niharnandala/tiny-aya-ntsb-reviewer-lora"
jobs = []
for part in ["1/3", "2/3", "3/3"]:
    p = part[0]
    jobs.append(run_uv_job(url, script_args=[f"before_p{p}", "none", "all300", part],
                           flavor="a100-large", timeout="3h", secrets={"HF_TOKEN": token}, token=token))
    jobs.append(run_uv_job(url, script_args=[f"v1_p{p}", V1, "exam100", part],
                           flavor="a100-large", timeout="3h", secrets={"HF_TOKEN": token}, token=token))
for j in jobs:
    print(j.url)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_experimental.py:59: UserWarning: 'HfApi.run_uv_job' is experimental and might be subject to breaking changes in the future without prior notice. You can disable this warning by setting `HF_HUB_DISABLE_EXPERIMENTAL_WARNING=1` as environment variable.
  warnings.warn(


https://huggingface.co/jobs/niharnandala/6afa98b723985fb9cd84482c
https://huggingface.co/jobs/niharnandala/6afa98b723985fb9cd84482e
https://huggingface.co/jobs/niharnandala/6afa98b823985fb9cd844830
https://huggingface.co/jobs/niharnandala/6afa98b829dd05ff0369c62f
https://huggingface.co/jobs/niharnandala/6afa98b829dd05ff0369c631
https://huggingface.co/jobs/niharnandala/6afa98b823985fb9cd844832


In [3]:
job = run_uv_job("https://raw.githubusercontent.com/niharnandala/ntsb-safety-review-agent/main/scripts/06_train_v2.py",
                 flavor="a100-large", timeout="2h", secrets={"HF_TOKEN": token}, token=token)
print(job.url)

https://huggingface.co/jobs/niharnandala/6afa9fa423985fb9cd844c0c


In [4]:
check = run_uv_job(url, script_args=["v2_check10", "niharnandala/tiny-aya-ntsb-reviewer-lora-v2", "exam100", "1/10"],
                   flavor="a100-large", timeout="1h", secrets={"HF_TOKEN": token}, token=token)
print(check.url)

https://huggingface.co/jobs/niharnandala/6afaa59f23985fb9cd844f2c


In [5]:
V2 = "niharnandala/tiny-aya-ntsb-reviewer-lora-v2"
v2_jobs = []
for part in ["1/3", "2/3", "3/3"]:
    p = part[0]
    v2_jobs.append(run_uv_job(url, script_args=[f"v2_p{p}", V2, "all300", part],
                              flavor="a100-large", timeout="3h", secrets={"HF_TOKEN": token}, token=token))
for j in v2_jobs:
    print(j.url)

https://huggingface.co/jobs/niharnandala/6afaa8ec29dd05ff0369d145
https://huggingface.co/jobs/niharnandala/6afaa8ed29dd05ff0369d147
https://huggingface.co/jobs/niharnandala/6afaa8ed29dd05ff0369d149


In [1]:
!pip install -q -U huggingface_hub
from google.colab import userdata
from huggingface_hub import HfApi

token = userdata.get("HF_TOKEN")
api = HfApi(token=token)

MODELS = ["niharnandala/tiny-aya-ntsb-reviewer-lora",
          "niharnandala/tiny-aya-ntsb-reviewer-lora-v2"]
DATASETS = ["niharnandala/ntsb-episodes",
            "niharnandala/ntsb-review-trajectories",
            "niharnandala/ntsb-review-trajectories-v2",
            "niharnandala/ntsb-eval-runs"]

# 1. make every repo public and not gated
for repo in MODELS:
    api.update_repo_settings(repo, private=False, gated=False, repo_type="model")
    print("public:", repo)
for repo in DATASETS:
    api.update_repo_settings(repo, private=False, gated=False, repo_type="dataset")
    print("public:", repo)

# 2. collection with the models
models_col = api.create_collection(
    title="NTSB Review Agent - Models", namespace="niharnandala", private=False, exists_ok=True,
    description="LoRA adapters for Tiny Aya Global: NTSB safety-recommendation review agent (v1, v2 final)")
for repo in MODELS:
    api.add_collection_item(models_col.slug, item_id=repo, item_type="model", exists_ok=True)

# 3. collection with the datasets
data_col = api.create_collection(
    title="NTSB Review Agent - Datasets", namespace="niharnandala", private=False, exists_ok=True,
    description="Episodes, training trajectories (v1, v2), and all evaluation runs with conversations, audit logs and scores")
for repo in DATASETS:
    api.add_collection_item(data_col.slug, item_id=repo, item_type="dataset", exists_ok=True)

print("\nMODELS COLLECTION:  ", "https://huggingface.co/collections/" + models_col.slug)
print("DATASETS COLLECTION:", "https://huggingface.co/collections/" + data_col.slug)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.4.0 which is incompatible.
public: niharnandala/tiny-aya-ntsb-reviewer-lora
public: niharnandala/tiny-aya-ntsb-reviewer-lora-v2
public: niharnandala/ntsb-episodes
public: niharnandala/ntsb-review-trajectories
public: niharnandala/ntsb-review-trajectories-v2
public: niharnandala/ntsb-eval-runs

MODELS COLLECTION:   https://huggingface.co/collections/niharnandala/ntsb-review-agent-models-6afad39c983f2538a5e03400
DATASETS COLLECTION: https://huggingface.co/collections/niharnandala/ntsb-review-agent-datasets-6afad39c680cb387d5ff8d69
